# 02. 確率・統計の復習 - 統計的多様体への準備

情報幾何では確率分布の集合を幾何学的対象として扱います。
本ノートブックでは、その準備として確率・統計の重要概念を復習します。

## 本ノートブックの目標
- パラメトリック分布族を「空間」として捉える視点を得る
- 指数型分布族の特別な性質を理解する
- 最尤推定とベイズ推定の幾何学的準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gammaln

plt.rcParams['figure.figsize'] = (10, 6)

---
## 1. パラメトリック分布族

### 定義
**パラメトリック分布族**とは、パラメータ $\theta \in \Theta$ で添字付けられた確率分布の集合：
$$\mathcal{S} = \{ p(x | \theta) : \theta \in \Theta \}$$

### 例
- 正規分布族: $\{ N(\mu, \sigma^2) : \mu \in \mathbb{R}, \sigma > 0 \}$ （2次元）
- ベルヌーイ分布族: $\{ \text{Bernoulli}(p) : p \in (0, 1) \}$ （1次元）
- ポアソン分布族: $\{ \text{Poisson}(\lambda) : \lambda > 0 \}$ （1次元）

**🔗 情報幾何の視点**: パラメータ空間 $\Theta$ が**統計的多様体**になる

In [ ]:
def visualize_distribution_family():
    """
    パラメトリック分布族を「空間」として可視化
    各パラメータ値が一つの確率分布に対応
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    x = np.linspace(-5, 10, 200)
    
    # 正規分布族（μを変化）
    ax1 = axes[0]
    sigma = 1
    for mu in [-2, -1, 0, 1, 2]:
        y = stats.norm.pdf(x, mu, sigma)
        ax1.plot(x, y, label=f'μ={mu}')
    ax1.set_xlabel('x')
    ax1.set_ylabel('p(x)')
    ax1.set_title('Gaussian family (varying μ, σ=1)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 正規分布族（σを変化）
    ax2 = axes[1]
    mu = 0
    for sigma in [0.5, 1, 1.5, 2, 3]:
        y = stats.norm.pdf(x, mu, sigma)
        ax2.plot(x, y, label=f'σ={sigma}')
    ax2.set_xlabel('x')
    ax2.set_ylabel('p(x)')
    ax2.set_title('Gaussian family (μ=0, varying σ)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # パラメータ空間
    ax3 = axes[2]
    mu_vals = np.linspace(-2, 2, 10)
    sigma_vals = np.linspace(0.5, 3, 10)
    for mu in mu_vals:
        for sigma in sigma_vals:
            ax3.plot(mu, sigma, 'b.', markersize=4)
    ax3.set_xlabel('μ')
    ax3.set_ylabel('σ')
    ax3.set_title('Parameter space Θ\n(Each point = one distribution)')
    ax3.axhline(0, color='k', linestyle='--', alpha=0.3)
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("""
【情報幾何の視点】
- 右図のパラメータ空間の各点が、一つの確率分布に対応
- この空間が「統計的多様体」
- 空間内の「距離」をどう測るか → Fisher計量
""")

visualize_distribution_family()

---
## 2. 指数型分布族（非常に重要！）

### 定義
**指数型分布族 (Exponential Family)** は以下の形で書ける分布族：
$$p(x | \theta) = \exp\left[ \theta^\top T(x) - \psi(\theta) + h(x) \right]$$

ここで：
- $\theta$: **自然パラメータ** (natural parameter)
- $T(x)$: **十分統計量** (sufficient statistic)
- $\psi(\theta)$: **対数分配関数** (log-partition function)
- $h(x)$: 基底測度

### なぜ重要か？
1. 多くの重要な分布が指数型に属する
2. 情報幾何で**双対平坦**という特別な構造を持つ
3. 共役事前分布が簡単に構成できる（ベイズ推定）

In [ ]:
def show_exponential_family_examples():
    """
    主要な指数型分布族の例を表示
    """
    examples = [
        {
            'name': 'Bernoulli(p)',
            'natural_param': 'θ = log(p/(1-p))',
            'sufficient_stat': 'T(x) = x',
            'log_partition': 'ψ(θ) = log(1 + eᶿ)',
            'mean_param': 'η = p = σ(θ)'
        },
        {
            'name': 'Gaussian(μ, σ²) [σ² known]',
            'natural_param': 'θ = μ/σ²',
            'sufficient_stat': 'T(x) = x',
            'log_partition': 'ψ(θ) = σ²θ²/2',
            'mean_param': 'η = μ'
        },
        {
            'name': 'Gaussian(μ, σ²) [both unknown]',
            'natural_param': 'θ = (μ/σ², -1/(2σ²))',
            'sufficient_stat': 'T(x) = (x, x²)',
            'log_partition': 'ψ(θ) = -θ₁²/(4θ₂) - ½log(-2θ₂)',
            'mean_param': 'η = (μ, μ² + σ²)'
        },
        {
            'name': 'Poisson(λ)',
            'natural_param': 'θ = log(λ)',
            'sufficient_stat': 'T(x) = x',
            'log_partition': 'ψ(θ) = eᶿ',
            'mean_param': 'η = λ'
        },
    ]
    
    print("=" * 70)
    print("指数型分布族の例")
    print("=" * 70)
    
    for ex in examples:
        print(f"\n【{ex['name']}】")
        print(f"  Natural parameter:  {ex['natural_param']}")
        print(f"  Sufficient stat:    {ex['sufficient_stat']}")
        print(f"  Log-partition:      {ex['log_partition']}")
        print(f"  Mean parameter:     {ex['mean_param']}")
    
    print("\n" + "=" * 70)
    print("""
【情報幾何との接続】
- 自然パラメータ θ と期待値パラメータ η は「双対座標」
- η = ∇ψ(θ)  という関係（Legendre変換）
- この双対性が情報幾何の中心的構造
""")

show_exponential_family_examples()

In [ ]:
def visualize_dual_coordinates():
    """
    ベルヌーイ分布での自然パラメータと期待値パラメータの関係を可視化
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # θ（自然パラメータ）と η（期待値パラメータ）の関係
    # Bernoulli: θ = log(p/(1-p)), η = p = 1/(1+e^{-θ})
    theta = np.linspace(-4, 4, 100)
    eta = 1 / (1 + np.exp(-theta))  # シグモイド関数
    
    ax1 = axes[0]
    ax1.plot(theta, eta, 'b-', linewidth=2)
    ax1.set_xlabel('θ (natural parameter)')
    ax1.set_ylabel('η (mean parameter = p)')
    ax1.set_title('Bernoulli: η = σ(θ) = 1/(1+e⁻ᶿ)')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0.5, color='r', linestyle='--', alpha=0.5)
    ax1.axvline(0, color='r', linestyle='--', alpha=0.5)
    
    # 対数分配関数 ψ(θ) = log(1 + e^θ)
    psi = np.log(1 + np.exp(theta))
    dpsi = 1 / (1 + np.exp(-theta))  # ψ'(θ) = η
    
    ax2 = axes[1]
    ax2.plot(theta, psi, 'g-', linewidth=2, label='ψ(θ) = log(1+eᶿ)')
    ax2.plot(theta, dpsi, 'b--', linewidth=2, label="ψ'(θ) = η")
    ax2.set_xlabel('θ')
    ax2.set_ylabel('ψ(θ)')
    ax2.set_title('Log-partition function and its derivative')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("""
【重要な関係式】
η = ∇ψ(θ)        ← 期待値パラメータは対数分配関数の勾配
Var[T(x)] = ∇²ψ(θ)  ← Fisher情報 = 対数分配関数のヘッセ行列

これが「指数型分布族は双対平坦」の数学的根拠
""")

visualize_dual_coordinates()

---
## 3. 最尤推定

### 定義
データ $x_1, \ldots, x_n$ が与えられたとき、**最尤推定量 (MLE)** は：
$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta \sum_{i=1}^n \log p(x_i | \theta)$$

### 指数型分布族での特殊性
指数型分布族では、MLEは**十分統計量の標本平均**と**期待値パラメータ**を一致させる：
$$\hat{\eta} = \frac{1}{n} \sum_{i=1}^n T(x_i)$$

**🔗 情報幾何の視点**: MLEは「m-射影」（期待値パラメータ座標での最近点）

In [ ]:
def demonstrate_mle_geometric():
    """
    最尤推定の幾何学的解釈を可視化
    """
    np.random.seed(42)
    
    # 真のパラメータ
    true_mu, true_sigma = 2.0, 1.5
    
    # データ生成
    n_samples = 50
    data = np.random.normal(true_mu, true_sigma, n_samples)
    
    # MLE
    mle_mu = np.mean(data)
    mle_sigma = np.std(data, ddof=0)  # MLEはn で割る
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左: 対数尤度関数
    ax1 = axes[0]
    mu_range = np.linspace(0, 4, 100)
    sigma_range = np.linspace(0.5, 3, 100)
    MU, SIGMA = np.meshgrid(mu_range, sigma_range)
    
    # 対数尤度
    log_likelihood = np.zeros_like(MU)
    for i in range(len(mu_range)):
        for j in range(len(sigma_range)):
            log_likelihood[j, i] = np.sum(stats.norm.logpdf(data, mu_range[i], sigma_range[j]))
    
    contour = ax1.contour(MU, SIGMA, log_likelihood, levels=20)
    ax1.plot(true_mu, true_sigma, 'g*', markersize=15, label=f'True ({true_mu}, {true_sigma})')
    ax1.plot(mle_mu, mle_sigma, 'r^', markersize=12, label=f'MLE ({mle_mu:.2f}, {mle_sigma:.2f})')
    ax1.set_xlabel('μ')
    ax1.set_ylabel('σ')
    ax1.set_title('Log-likelihood contours')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右: パラメータ空間での軌跡（最適化過程のイメージ）
    ax2 = axes[1]
    
    # 初期値から最適解への「経路」（概念的）
    init_mu, init_sigma = 0.5, 2.5
    t = np.linspace(0, 1, 20)
    path_mu = init_mu + (mle_mu - init_mu) * t
    path_sigma = init_sigma + (mle_sigma - init_sigma) * t
    
    ax2.plot(path_mu, path_sigma, 'b.-', markersize=8, label='Optimization path')
    ax2.plot(init_mu, init_sigma, 'bs', markersize=12, label='Initial')
    ax2.plot(mle_mu, mle_sigma, 'r^', markersize=12, label='MLE')
    ax2.plot(true_mu, true_sigma, 'g*', markersize=15, label='True')
    
    ax2.set_xlabel('μ')
    ax2.set_ylabel('σ')
    ax2.set_title('Parameter space trajectory')
    ax2.legend()
    ax2.set_xlim(-0.5, 4.5)
    ax2.set_ylim(0, 3.5)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"""
【結果】
- 真のパラメータ: μ={true_mu}, σ={true_sigma}
- MLE: μ̂={mle_mu:.3f}, σ̂={mle_sigma:.3f}
- 十分統計量: T̄ = (x̄, s²) = ({np.mean(data):.3f}, {np.var(data):.3f})

【情報幾何の視点】
- MLEはデータの経験分布から統計的多様体への「射影」
- 期待値パラメータ座標では、この射影は直交射影になる
""")

demonstrate_mle_geometric()

---
## 4. ベイズ推定

### ベイズの定理
$$p(\theta | x) = \frac{p(x | \theta) p(\theta)}{p(x)} \propto p(x | \theta) p(\theta)$$

- $p(\theta)$: 事前分布
- $p(x | \theta)$: 尤度
- $p(\theta | x)$: 事後分布

### 共役事前分布
事後分布が事前分布と同じ形になる事前分布を**共役事前分布**という。

**🔗 情報幾何の視点**: ベイズ更新は統計的多様体上の「移動」として解釈できる

In [ ]:
def demonstrate_bayesian_update():
    """
    正規分布の平均のベイズ推定を可視化
    （既知の分散、正規事前分布の場合）
    """
    # 設定
    sigma_known = 1.0  # 既知の標準偏差
    
    # 事前分布: N(μ_0, τ_0²)
    mu_prior = 0.0
    tau_prior = 2.0  # 事前の不確実性（大きい）
    
    # 観測データ
    observations = [1.5, 2.0, 1.8, 2.2, 1.9]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左図: 事後分布の逐次更新
    ax1 = axes[0]
    mu_range = np.linspace(-3, 5, 200)
    
    # 事前分布
    prior = stats.norm.pdf(mu_range, mu_prior, tau_prior)
    ax1.plot(mu_range, prior, 'k--', linewidth=2, label='Prior')
    
    # 逐次更新
    mu_post = mu_prior
    precision_post = 1 / tau_prior**2
    
    colors = plt.cm.Blues(np.linspace(0.3, 1, len(observations)))
    
    for i, x in enumerate(observations):
        # ベイズ更新（正規-正規の共役）
        precision_likelihood = 1 / sigma_known**2
        precision_post_new = precision_post + precision_likelihood
        mu_post_new = (precision_post * mu_post + precision_likelihood * x) / precision_post_new
        
        precision_post = precision_post_new
        mu_post = mu_post_new
        tau_post = 1 / np.sqrt(precision_post)
        
        posterior = stats.norm.pdf(mu_range, mu_post, tau_post)
        ax1.plot(mu_range, posterior, color=colors[i], linewidth=1.5,
                 label=f'After obs {i+1}: x={x}')
    
    ax1.axvline(np.mean(observations), color='r', linestyle=':', label=f'Sample mean={np.mean(observations):.2f}')
    ax1.set_xlabel('μ')
    ax1.set_ylabel('p(μ)')
    ax1.set_title('Bayesian update of Gaussian mean')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # 右図: パラメータ空間での軌跡
    ax2 = axes[1]
    
    # 更新の軌跡を記録
    mu_post = mu_prior
    precision_post = 1 / tau_prior**2
    trajectory = [(mu_post, 1/np.sqrt(precision_post))]
    
    for x in observations:
        precision_likelihood = 1 / sigma_known**2
        precision_post_new = precision_post + precision_likelihood
        mu_post_new = (precision_post * mu_post + precision_likelihood * x) / precision_post_new
        
        precision_post = precision_post_new
        mu_post = mu_post_new
        tau_post = 1 / np.sqrt(precision_post)
        
        trajectory.append((mu_post, tau_post))
    
    trajectory = np.array(trajectory)
    ax2.plot(trajectory[:, 0], trajectory[:, 1], 'b.-', markersize=10, linewidth=2)
    ax2.plot(trajectory[0, 0], trajectory[0, 1], 'ks', markersize=12, label='Prior')
    ax2.plot(trajectory[-1, 0], trajectory[-1, 1], 'r^', markersize=12, label='Final posterior')
    
    # 観測点をプロット（「方向」として）
    for i, x in enumerate(observations):
        ax2.axvline(x, color='gray', linestyle='--', alpha=0.3)
    
    ax2.set_xlabel('μ (posterior mean)')
    ax2.set_ylabel('τ (posterior std)')
    ax2.set_title('Trajectory in parameter space')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"""
【ベイズ更新の軌跡】
- 事前: μ={mu_prior}, τ={tau_prior}
- 事後: μ={trajectory[-1, 0]:.3f}, τ={trajectory[-1, 1]:.3f}

【情報幾何の視点】
- 右図の軌跡は統計的多様体上の「曲線」
- 各更新は「現在の信念」から「観測方向」への移動
- カルマンフィルタも同様の幾何学的構造を持つ
""")

demonstrate_bayesian_update()

---
## 5. 確認問題

### Q1. 指数型分布族
ポアソン分布 $p(x|\lambda) = \frac{\lambda^x e^{-\lambda}}{x!}$ を指数型分布族の標準形に書き直せ。

### Q2. 十分統計量
正規分布 $N(\mu, \sigma^2)$（両方未知）の十分統計量は何か？

### Q3. ベイズ更新
ベルヌーイ尤度に対する共役事前分布は何か？更新則を導け。

In [ ]:
# Q1の解答
print("""
【Q1 解答】
p(x|λ) = λˣ e⁻ᵏ / x!
       = exp(x log λ - λ - log x!)

標準形: p(x|θ) = exp(θT(x) - ψ(θ) + h(x))

- θ = log λ  （自然パラメータ）
- T(x) = x   （十分統計量）
- ψ(θ) = eᶿ = λ  （対数分配関数）
- h(x) = -log x!  （基底測度）

期待値パラメータ: η = ∇ψ(θ) = eᶿ = λ
""")

In [ ]:
# Q2の解答
print("""
【Q2 解答】
正規分布 N(μ, σ²) の十分統計量は:

T(x₁, ..., xₙ) = (Σxᵢ, Σxᵢ²) または等価的に (x̄, s²)

- Σxᵢ は μ の推定に使う
- Σxᵢ² は σ² の推定に使う

これ以外の情報（個々のデータの順序など）は推定に不要。
""")

In [ ]:
# Q3の解答
print("""
【Q3 解答】
ベルヌーイ尤度の共役事前分布は Beta分布:

事前: p ~ Beta(α, β)
尤度: x₁,...,xₙ ~ Bernoulli(p)
事後: p|data ~ Beta(α + Σxᵢ, β + n - Σxᵢ)

更新則:
  α_new = α + (成功数)
  β_new = β + (失敗数)

事後平均: E[p|data] = α_new / (α_new + β_new)
""")

---
## まとめ

| 確率・統計の概念 | 情報幾何での対応 |
|----------------|----------------|
| パラメトリック分布族 | 統計的多様体 |
| 指数型分布族 | 双対平坦多様体 |
| 自然パラメータ θ | e-座標（指数座標） |
| 期待値パラメータ η | m-座標（混合座標） |
| 対数分配関数 ψ(θ) | ポテンシャル関数 |
| 最尤推定 | m-射影 |
| ベイズ更新 | 多様体上の曲線 |

---
**次のノートブック**: `03_kl_divergence_fisher.ipynb` - KLダイバージェンスとFisher情報